In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

df_ml = pd.read_csv("../data/df_master_financials.csv")

target = df_ml['Churn'].map({'Yes': 1, 'No': 0})

features_to_drop = ['customerID', 'Churn']
X_raw = df_ml.drop(columns=features_to_drop)

X = pd.get_dummies(X_raw, drop_first=True)

# 80% Train, 20% Test with Stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=0.2, random_state=42, stratify=target
)

# Model Training (Random Forest Classifier)
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

# Evaluate Baseline Performance
y_pred = model.predict(X_test)
print("\n--- Model Performance Evaluation ---")
print(f"ROC-AUC Score: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f}")
print(classification_report(y_test, y_pred))

df_ml['Churn_Probability'] = model.predict_proba(X)[:, 1]

# Financial Loss Metric: Expected Revenue Loss ($)
df_ml['Expected_Revenue_Loss'] = df_ml['Churn_Probability'] * df_ml['MonthlyCharges']

# Filter for Active Customers & Rank by Top Dollar Loss Risk
active_at_risk = df_ml[df_ml['Churn'] == 'No'].sort_values(by='Expected_Revenue_Loss', ascending=False)

print("\n--- Top 5 Active Accounts at Highest Financial Risk ---")
active_at_risk[['customerID', 'Contract', 'MonthlyCharges', 'Churn_Probability', 'Expected_Revenue_Loss']].head()

df_ml.to_csv("../data/df_master_final_risk.csv", index=False)


--- Model Performance Evaluation ---
ROC-AUC Score: 0.8541
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409


--- Top 5 Active Accounts at Highest Financial Risk ---
